In [ ]:
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import auc

sys.path.append(os.path.abspath("../"))

from util import dataset as u_dataset

In [ ]:
log_path = Path("../../logs/fit/")

In [ ]:
def plot_pr_curve(run: int, mode: str, specification: str, object_name: str):
    # Your base directory
    base_dir = Path(log_path, mode, f"run_{run}", specification)

    # Get the single folder inside base_dir
    folders = [f for f in base_dir.iterdir() if f.is_dir()]
    if len(folders) == 1:
        path_to_model = folders[0]
    else:
        raise ValueError(f"Expected exactly one folder in {base_dir}, found {len(folders)}")

    # Load data
    recalls = np.load(path_to_model.as_posix() + f"/recalls/{object_name}.npy")
    precisions = np.load(path_to_model.as_posix() + f"/precisions/{object_name}.npy")

    # # Compute AUC-PR
    pr_auc = auc(recalls, precisions)

    # Plot
    fig, ax = plt.subplots(figsize=(8, 6))

    ax.plot(
        recalls,
        precisions,
        color="steelblue",
        linewidth=2,
        label=f"PR Curve (AUC = {pr_auc:.3f})",
    )
    ax.fill_between(recalls, precisions, alpha=0.1, color="steelblue")

    # Baseline (random classifier)
    # ax.axhline(
    #     y=precisions.mean(),
    #     color="gray",
    #     linestyle="--",
    #     linewidth=1,
    #     label="Baseline (random)",
    # )

    ax.set_xlabel("Recall", fontsize=13)
    ax.set_ylabel("Precision", fontsize=13)
    ax.set_title(f"Precision-Recall Curve, run: {run}, object: {object_name}", fontsize=15)
    ax.set_xlim([0.0, 1.0])
    ax.set_ylim([0.0, 1.05])
    ax.legend(loc="upper right", fontsize=11)
    ax.grid(True, linestyle="--", alpha=0.5)

    plt.tight_layout()
    plt.savefig("precision_recall_curve.png", dpi=150)
    plt.show()

In [ ]:
def plot_curves(run, mode, specification):
    for object in u_dataset.CategoryNames:
        if object == u_dataset.CategoryNames.BALL:
            plot_pr_curve(run, mode, specification, object.value)

        if object == u_dataset.CategoryNames.PENALTYMARK:
            plot_pr_curve(run, mode, specification, object.value)

        if object == u_dataset.CategoryNames.INTERSECTIONS:
            for type in list(u_dataset.IntersectionType)[1:]:
                object_name = f"{object.value}_{type.name}"
                plot_pr_curve(run, mode, specification, object_name)

In [ ]:
mode = "classifier-basic"
specification = "v1"

for run in range(1, 4):
    plot_curves(run, mode, specification)